# Export RS3 Hsu2013-only train, test, and unseen sequences

Place this notebook in `rs_dev/code` and run all cells.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from datasets import dataset_list

TRACR_FILTER = "Hsu2013"
SPLIT_SEED = 42
PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../models/rs3_hsu2013_fixed_split_100_trials/split_sequences")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_NAMES_FILE = PROCESSED_DIR / "train_data_names.csv"

print("tracr:", TRACR_FILTER)
print("Output:", OUTPUT_DIR.resolve())


tracr: Hsu2013
Output: /Users/zhangjiongyu/rs_dev/models/rs3_hsu2013_fixed_split_100_trials/split_sequences


In [3]:
train_data_names = pd.read_csv(TRAIN_NAMES_FILE)["name"].dropna().astype(str).tolist()
train_data_list = [ds for ds in dataset_list if ds.name in train_data_names]

for ds in train_data_list:
    ds.load_data()
    ds.set_sgrnas()

sg_df_list = []
for ds in train_data_list:
    df = ds.get_sg_df(include_group=True, include_activity=True).copy()
    df["dataset"] = ds.name
    df["tracr"] = ds.tracr
    sg_df_list.append(df)

groups = (
    pd.concat(sg_df_list, ignore_index=True)
    .groupby("sgRNA Context Sequence", as_index=False)
    .agg(target=("sgRNA Target", lambda x: ", ".join(sorted({
        str(v).upper() for v in x
        if not pd.isna(v) and str(v).strip() != ""
    }))))
)

groups["target"] = groups.apply(
    lambda r: r["target"] if r["target"] != "" else r["sgRNA Context Sequence"],
    axis=1,
)

all_data = (
    pd.concat(sg_df_list, ignore_index=True)
    .merge(groups[["sgRNA Context Sequence", "target"]],
           on="sgRNA Context Sequence", how="inner")
    .sort_values(["dataset", "target"])
    .reset_index(drop=True)
)

all_data["sgRNA Activity"] = pd.to_numeric(all_data["sgRNA Activity"], errors="coerce")
all_data = all_data.dropna(subset=[
    "sgRNA Sequence", "sgRNA Context Sequence", "sgRNA Activity",
    "dataset", "tracr", "target"
]).reset_index(drop=True)

filtered_data = all_data.loc[
    all_data["tracr"].astype(str) == TRACR_FILTER
].copy().reset_index(drop=True)

if filtered_data.empty:
    raise ValueError(f"No rows found for tracr={TRACR_FILTER!r}")

print("Filtered rows:", len(filtered_data))
display(filtered_data["dataset"].value_counts().rename("n").to_frame())


Filtered rows: 29951


,n
dataset,
Kim2019_train,12832
Xiang2021,11397
Doench2016,2536
Doench2014_mouse,1169
Wang2014,1022
Doench2014_human,995


In [5]:
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
seen_idx, unseen_idx = next(
    outer.split(filtered_data, y=filtered_data["dataset"], groups=filtered_data["target"])
)

seen_data = filtered_data.iloc[seen_idx].reset_index(drop=True)
unseen_data = filtered_data.iloc[unseen_idx].reset_index(drop=True)

inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED)
train_idx, test_idx = next(
    inner.split(seen_data, y=seen_data["dataset"], groups=seen_data["target"])
)

train_data = seen_data.iloc[train_idx].reset_index(drop=True)
test_data = seen_data.iloc[test_idx].reset_index(drop=True)

assert set(train_data["target"]).isdisjoint(set(test_data["target"]))
assert set(train_data["target"]).isdisjoint(set(unseen_data["target"]))
assert set(test_data["target"]).isdisjoint(set(unseen_data["target"]))

display(pd.DataFrame({
    "subset": ["train", "test", "unseen"],
    "n_rows": [len(train_data), len(test_data), len(unseen_data)],
    "fraction": [
        len(train_data)/len(filtered_data),
        len(test_data)/len(filtered_data),
        len(unseen_data)/len(filtered_data),
    ]
}))


,subset,n_rows,fraction
0,train,20949,0.699442
1,test,4119,0.137525
2,unseen,4883,0.163033


In [7]:
def prepare_export(df):
    out = df.copy().reset_index(drop=True)
    out["sgRNA_sequence"] = out["sgRNA Sequence"].astype(str).str.strip().str.upper()
    out["target_context_sequence"] = (
        out["sgRNA Context Sequence"].astype(str).str.strip().str.upper()
    )

    lengths = out["target_context_sequence"].str.len()
    out["target_protospacer_20nt"] = np.where(
        lengths >= 27,
        out["target_context_sequence"].str.slice(4, 24),
        out["sgRNA_sequence"],
    )
    out["pam_sequence"] = np.where(
        lengths >= 27,
        out["target_context_sequence"].str.slice(-6, -3),
        "",
    )
    out["activity"] = pd.to_numeric(out["sgRNA Activity"], errors="coerce")

    metadata = [c for c in ["dataset", "tracr", "target", "sgRNA Target"] if c in out.columns]
    out = out[
        ["sgRNA_sequence", "target_protospacer_20nt",
         "target_context_sequence", "pam_sequence", "activity"] + metadata
    ]

    return out.dropna(
        subset=["sgRNA_sequence", "target_context_sequence", "activity"]
    ).reset_index(drop=True)


def save_series(series, path):
    series.to_csv(path, index=False, header=False)


def export_subset(df, subset_name):
    export_df = prepare_export(df)
    subset_dir = OUTPUT_DIR / subset_name
    subset_dir.mkdir(parents=True, exist_ok=True)

    save_series(export_df["sgRNA_sequence"],
                subset_dir / f"{subset_name}_sgRNA_sequences.txt")
    save_series(export_df["target_protospacer_20nt"],
                subset_dir / f"{subset_name}_target_20nt.txt")
    save_series(export_df["target_context_sequence"],
                subset_dir / f"{subset_name}_target_context_full.txt")
    save_series(export_df["pam_sequence"],
                subset_dir / f"{subset_name}_pam.txt")
    save_series(export_df["activity"],
                subset_dir / f"{subset_name}_activity.txt")

    export_df.to_csv(
        subset_dir / f"{subset_name}_complete.tsv",
        sep="\t",
        index=False,
    )

    print(subset_name, len(export_df), "rows")
    return export_df


train_export = export_subset(train_data, "train")
test_export = export_subset(test_data, "test")
unseen_export = export_subset(unseen_data, "unseen")


train 20949 rows
test 4119 rows
unseen 4883 rows


In [9]:
def count_lines(path):
    with open(path, "r") as f:
        return sum(1 for line in f if line.strip())

for subset_name, export_df in [
    ("train", train_export),
    ("test", test_export),
    ("unseen", unseen_export),
]:
    subset_dir = OUTPUT_DIR / subset_name
    files = [
        subset_dir / f"{subset_name}_sgRNA_sequences.txt",
        subset_dir / f"{subset_name}_target_20nt.txt",
        subset_dir / f"{subset_name}_target_context_full.txt",
        subset_dir / f"{subset_name}_pam.txt",
        subset_dir / f"{subset_name}_activity.txt",
    ]
    counts = [count_lines(p) for p in files]
    print(subset_name, counts)
    assert len(set(counts)) == 1

print("All files are aligned.")


train [20949, 20949, 20949, 20949, 20949]
test [4119, 4119, 4119, 4119, 4119]
unseen [4883, 4883, 4883, 4883, 4883]
All files are aligned.
